# Résumé automatique de papiers arXiv (cs.AI)
## Pipeline hybride : PEGASUS-arxiv + fallback TextRank

Ce notebook suit les étapes du pipeline de résumé :
1. Récupération des papiers depuis l'API arXiv
2. Parsing PDF avec détection de sections
3. Résumé section par section avec PEGASUS-arxiv
4. Fallback TextRank si la détection échoue
5. Composition du sommaire final

## Cellule 0 — Installation des dépendances

Décommentez si nécessaire.

In [1]:
# !pip install arxiv pymupdf transformers torch sumy nltk sentencepiece protobuf

## Étape 1 — Imports et configuration

In [2]:
from __future__ import annotations
import re
import logging
from dataclasses import dataclass, field
from pathlib import Path
from IPython.display import display, Markdown

import arxiv
import fitz  # PyMuPDF

logging.basicConfig(level=logging.WARNING)  # on réduit le bruit des logs dans le notebook

# --- Paramètres modifiables ---
QUERY      = "cat:cs.AI"   # requête arXiv
MAX_PAPERS = 2             # nombre de papiers à traiter (garder petit pour les tests)
OUTPUT_DIR = Path("./papers")
SUMMARY_DIR = Path("./summaries")

print("✓ Imports OK")
print(f"  Requête    : {QUERY}")
print(f"  Nb papiers : {MAX_PAPERS}")
print(f"  Dossier PDF : {OUTPUT_DIR}")

✓ Imports OK
  Requête    : cat:cs.AI
  Nb papiers : 2
  Dossier PDF : papers


## Étape 2 — Récupération des papiers depuis l'API arXiv

On interroge l'API officielle arXiv pour obtenir les `MAX_PAPERS` papiers les plus récents, puis on télécharge leurs PDF localement.

In [3]:
@dataclass
class PaperMeta:
    arxiv_id: str
    title: str
    authors: list[str]
    published: str
    pdf_url: str
    pdf_path: Path | None = None


def fetch_arxiv_papers(query: str, max_results: int, output_dir: Path) -> list[PaperMeta]:
    output_dir.mkdir(exist_ok=True, parents=True)
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )
    papers = []
    for result in search.results():
        meta = PaperMeta(
            arxiv_id=result.entry_id.split("/")[-1],
            title=result.title.strip(),
            authors=[a.name for a in result.authors],
            published=result.published.isoformat(),
            pdf_url=result.pdf_url,
        )
        meta.pdf_path = output_dir / f"{meta.arxiv_id}.pdf"
        if not meta.pdf_path.exists():
            print(f"  ⬇ Téléchargement : {meta.arxiv_id}")
            result.download_pdf(dirpath=str(output_dir), filename=meta.pdf_path.name)
        else:
            print(f"  ✓ Déjà en cache  : {meta.arxiv_id}")
        papers.append(meta)
    return papers


# --- Exécution ---
print(f"Recherche arXiv : '{QUERY}' — {MAX_PAPERS} papiers\n")
papers = fetch_arxiv_papers(QUERY, MAX_PAPERS, OUTPUT_DIR)

print(f"\n{len(papers)} papier(s) récupéré(s) :")
for i, p in enumerate(papers, 1):
    authors_str = ", ".join(p.authors[:3]) + (" et al." if len(p.authors) > 3 else "")
    print(f"\n  [{i}] {p.title}")
    print(f"       Auteurs : {authors_str}")
    print(f"       Date    : {p.published[:10]}")
    print(f"       ID      : {p.arxiv_id}")

Recherche arXiv : 'cat:cs.AI' — 2 papiers



C:\Users\nikol\AppData\Local\Temp\ipykernel_31208\3719896176.py:20: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


  ✓ Déjà en cache  : 2605.04039v1
  ✓ Déjà en cache  : 2605.04036v1

2 papier(s) récupéré(s) :

  [1] Safety and accuracy follow different scaling laws in clinical large language models
       Auteurs : Sebastian Wind, Tri-Thien Nguyen, Jeta Sopa et al.
       Date    : 2026-05-05
       ID      : 2605.04039v1

  [2] OpenSeeker-v2: Pushing the Limits of Search Agents with Informative and High-Difficulty Trajectories
       Auteurs : Yuwen Du, Rui Ye, Shuo Tang et al.
       Date    : 2026-05-05
       ID      : 2605.04036v1


## Étape 3 — Parsing PDF et détection de sections

On utilise **PyMuPDF** (`fitz`) qui expose les métadonnées de police, ce qui permet de repérer les en-têtes de section (lignes courtes avec une police plus grande que le corps du texte).

In [4]:
SECTION_KEYWORDS = {
    "abstract":      ["abstract"],
    "introduction":  ["introduction", "1. introduction"],
    "method":        ["method", "methodology", "approach", "model", "our approach", "proposed"],
    "results":       ["results", "experiments", "evaluation", "experimental results"],
    "conclusion":    ["conclusion", "conclusions", "discussion", "discussion and conclusion"],
}
SUMMARY_SECTIONS = ["introduction", "method", "results", "conclusion"]


@dataclass
class ParsedPaper:
    meta: PaperMeta
    full_text: str
    sections: dict[str, str] = field(default_factory=dict)


def _split_by_headers(full_text: str, headers: list) -> dict[str, str]:
    sections: dict[str, str] = {}
    text_lower = full_text.lower()
    matches = []
    for sec_name, keywords in SECTION_KEYWORDS.items():
        for kw in keywords:
            pattern = rf"\n\s*\d?\.?\s*{re.escape(kw)}\s*\n"
            m = re.search(pattern, text_lower)
            if m:
                matches.append((m.start(), sec_name))
                break
    matches.sort(key=lambda x: x[0])
    for i, (start, name) in enumerate(matches):
        end = matches[i + 1][0] if i + 1 < len(matches) else len(full_text)
        sections[name] = full_text[start:end].strip()
    return sections


def parse_pdf(meta: PaperMeta) -> ParsedPaper:
    doc = fitz.open(meta.pdf_path)
    full_text_parts = []
    candidate_headers = []

    for page_idx, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        for block in blocks:
            if block.get("type") != 0:
                continue
            for line in block["lines"]:
                line_text = " ".join(span["text"] for span in line["spans"]).strip()
                if not line_text:
                    continue
                full_text_parts.append(line_text)
                max_font_size = max(span["size"] for span in line["spans"])
                if len(line_text) < 80 and max_font_size > 11:
                    candidate_headers.append((page_idx, max_font_size, line_text))

    full_text = "\n".join(full_text_parts)
    sections = _split_by_headers(full_text, candidate_headers)
    return ParsedPaper(meta=meta, full_text=full_text, sections=sections)


# --- Exécution + affichage diagnostique ---
parsed_papers = []
for paper in papers:
    print(f"📄 Parsing : {paper.title[:70]}")
    parsed = parse_pdf(paper)
    parsed_papers.append(parsed)

    nb_chars = len(parsed.full_text)
    detected = list(parsed.sections.keys())
    print(f"   Texte extrait    : {nb_chars:,} caractères ({nb_chars // 5:,} mots ~)")
    print(f"   Sections trouvées: {detected if detected else '❌ aucune'}")

    # Aperçu du texte brut (100 premiers caractères de chaque section)
    for sec, content in parsed.sections.items():
        preview = content[:120].replace("\n", " ")
        print(f"   [{sec:12s}] {preview}…")
    print()

📄 Parsing : Safety and accuracy follow different scaling laws in clinical large la
   Texte extrait    : 186,610 caractères (37,322 mots ~)
   Sections trouvées: ['introduction', 'results', 'conclusion']
   [introduction] Introduction Large   language   models   (LLMs)   are   increasingly   being   evaluated   for   clinical   decision   s…
   [results     ] Results All analyses use the pooled 200-question RadSaFE-200 benchmark, with no stratification by source subset.   For  …
   [conclusion  ] Discussion Using   SaFE-Scale   on   RadSaFE-200,   this   study   shows   that   clinical   LLM   safety   cannot   be …

📄 Parsing : OpenSeeker-v2: Pushing the Limits of Search Agents with Informative an
   Texte extrait    : 23,525 caractères (4,705 mots ~)
   Sections trouvées: ['abstract', 'introduction', 'method', 'conclusion']
   [abstract    ] Abstract Deep search capabilities have become an indispensable competency for frontier Large Language Model (LLM) agents…
   [introduction] 1 In

## Étape 4 — Chargement du modèle PEGASUS-arxiv

Le modèle est chargé **une seule fois** (quelques secondes). Il sera réutilisé pour tous les papiers.

In [5]:
import time
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

MODEL_NAME = "google/pegasus-arxiv"

print(f"Chargement de {MODEL_NAME}...")
t0 = time.time()
tokenizer = PegasusTokenizer.from_pretrained(MODEL_NAME)
model     = PegasusForConditionalGeneration.from_pretrained(MODEL_NAME)
print(f"✓ Modèle chargé en {time.time() - t0:.1f}s")
print(f"  Paramètres : {model.num_parameters():,}")
print(f"  Fenêtre max : 1024 tokens")

C:\Users\nikol\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement de google/pegasus-arxiv...


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 29005.07it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-arxiv
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Modèle chargé en 11.5s
  Paramètres : 570,797,056
  Fenêtre max : 1024 tokens


## Étape 5 — Résumé des sections

### Logique de résumé
- **Texte court** (≤ 1024 tokens) → résumé direct avec PEGASUS
- **Texte long** → découpage en chunks → résumé de chaque chunk → re-résumé (map-reduce)
- **Fallback** : si moins de 2 sections détectées → résumé extractif **TextRank** (sumy)

In [6]:
MAX_INPUT_TOKENS = 1024


def summarize_text(text: str, max_target_tokens: int = 100) -> str:
    """Résume un texte qui tient en une fenêtre PEGASUS."""
    inputs = tokenizer(text, max_length=MAX_INPUT_TOKENS, truncation=True, return_tensors="pt")
    outputs = model.generate(
        **inputs, max_length=max_target_tokens,
        num_beams=4, length_penalty=2.0, early_stopping=True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def summarize_long(text: str, max_target_tokens: int = 120) -> tuple[str, str]:
    """Résume un texte long. Retourne (résumé, méthode utilisée)."""
    token_count = len(tokenizer.encode(text))
    if token_count <= MAX_INPUT_TOKENS:
        return summarize_text(text, max_target_tokens), "direct"

    # Chunking par phrases
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current, current_len = [], [], 0
    for s in sentences:
        s_len = len(tokenizer.encode(s, add_special_tokens=False))
        if current_len + s_len > 800 and current:
            chunks.append(" ".join(current))
            current, current_len = [s], s_len
        else:
            current.append(s)
            current_len += s_len
    if current:
        chunks.append(" ".join(current))

    chunk_summaries = [summarize_text(c, 80) for c in chunks]
    return summarize_text(" ".join(chunk_summaries), max_target_tokens), f"map-reduce ({len(chunks)} chunks)"


def textrank_fallback(text: str, num_sentences: int = 6) -> str:
    from sumy.parsers.plaintext import PlaintextParser
    from sumy.nlp.tokenizers import Tokenizer
    from sumy.summarizers.text_rank import TextRankSummarizer
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = TextRankSummarizer()
    return " ".join(str(s) for s in summarizer(parser.document, num_sentences))


# --- Résumé section par section avec affichage ---
LABELS = {
    "introduction": "Problème abordé",
    "method":       "Méthode proposée",
    "results":      "Principaux résultats",
    "conclusion":   "Conclusion / implications",
}

all_summaries = []   # [(meta, section_summaries_dict, structured)]

for parsed in parsed_papers:
    print(f"\n{'='*70}")
    print(f"📰 {parsed.meta.title[:70]}")
    print(f"{'='*70}")

    detected = [s for s in SUMMARY_SECTIONS if s in parsed.sections]
    print(f"  Sections utilisables : {detected}")

    if len(detected) < 2:
        print("  ⚠ Moins de 2 sections → fallback TextRank")
        body = textrank_fallback(parsed.full_text, num_sentences=6)
        all_summaries.append((parsed.meta, body, False))
        continue

    section_summaries = {}
    for sec in detected:
        text = parsed.sections[sec]
        n_words = len(text.split())
        if n_words < 30:
            print(f"  [{sec}] trop courte ({n_words} mots), skippée")
            continue
        t0 = time.time()
        summary, method = summarize_long(text)
        elapsed = time.time() - t0
        section_summaries[sec] = summary
        print(f"  [{sec:12s}] {n_words:4d} mots → résumé via {method} ({elapsed:.1f}s)")
        print(f"             {summary[:100]}…")

    all_summaries.append((parsed.meta, section_summaries, True))

print("\n✓ Résumés générés pour tous les papiers")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1259 > 1024). Running this sequence through the model will result in indexing errors



📰 Safety and accuracy follow different scaling laws in clinical large la
  Sections utilisables : ['introduction', 'results', 'conclusion']
  [introduction]  940 mots → résumé via map-reduce (2 chunks) (56.8s)
             language models are increasingly being evaluated for clinical decision support , evidence - grounded…
  [results     ] 6937 mots → résumé via map-reduce (14 chunks) (296.5s)
             we report the results of a pilot study to test the accuracy of clinical models used in the evaluatio…
  [conclusion  ] 15483 mots → résumé via map-reduce (34 chunks) (648.8s)
             in this paper , we present a new approach to the problem of option retrieval in biological systems .…

📰 OpenSeeker-v2: Pushing the Limits of Search Agents with Informative an
  Sections utilisables : ['introduction', 'method', 'conclusion']
  [introduction]  531 mots → résumé via direct (25.9s)
             in this paper , we present a new algorithm for the search of the most probable outcome of a

## Étape 6 — Affichage et sauvegarde des sommaires

Les sommaires sont affichés en Markdown directement dans le notebook, puis sauvegardés en fichiers `.md`.

In [7]:
def build_summary_md(meta: PaperMeta, body, structured: bool) -> str:
    authors_str = ", ".join(meta.authors[:3]) + (" et al." if len(meta.authors) > 3 else "")
    lines = [
        f"# {meta.title}",
        f"**Auteurs :** {authors_str}",
        f"**Date :** {meta.published[:10]}",
        f"**arXiv :** [{meta.arxiv_id}](https://arxiv.org/abs/{meta.arxiv_id})",
        "",
        "## Sommaire",
    ]
    if structured:
        for sec, summary in body.items():
            lines.append(f"\n### {LABELS.get(sec, sec.title())}")
            lines.append(summary)
    else:
        lines.append("\n> *Résumé extractif par TextRank — détection de sections insuffisante*")
        lines.append(body)
    return "\n".join(lines)


SUMMARY_DIR.mkdir(exist_ok=True, parents=True)

for meta, body, structured in all_summaries:
    md = build_summary_md(meta, body, structured)

    # Affichage dans le notebook
    display(Markdown("---"))
    display(Markdown(md))

    # Sauvegarde fichier
    out_file = SUMMARY_DIR / f"{meta.arxiv_id}_summary.md"
    out_file.write_text(md, encoding="utf-8")
    print(f"💾 Sauvegardé : {out_file}")

---

# Safety and accuracy follow different scaling laws in clinical large language models
**Auteurs :** Sebastian Wind, Tri-Thien Nguyen, Jeta Sopa et al.
**Date :** 2026-05-05
**arXiv :** [2605.04039v1](https://arxiv.org/abs/2605.04039v1)

## Sommaire

### Problème abordé
language models are increasingly being evaluated for clinical decision support , evidence - grounded reasoning , education and retrieval . <n> practice , however , is often driven by scaling : larger models are preferred over smaller ones , longer contexts are assumed to provide more complete evidence , additional inference-time is used to improve answer [ 1 , 2 , 3 ] . in this paper <n> , we propose a new measure , _ _ _ _ _ _ _ _ _ _ _ _

### Principaux résultats
we report the results of a pilot study to test the accuracy of clinical models used in the evaluation of deployed medical units . <n> the results show that clinical models are accurate to within a few percent over a wide range of deployment conditions . <n> we examined how reported confidence on correct , high - risk and unsafe outputs remained well below the median confidence threshold for clinical deployment . <n> the median confidence threshold remained well below the median confidence threshold for all four classes under every deployment condition , with the median confidence increases on high- and unsafe outrisks , putting only

### Conclusion / implications
in this paper , we present a new approach to the problem of option retrieval in biological systems . <n> we show that majority - vote analysis ( mwa ) is not sufficient to determine whether a given response is statistically significant . <n> we demonstrate that the the failure of an ensemble consisting of a large number of non - identical members is considered . <n> the failure of the ensemble was evaluated under three conditions : ( i ) the number of non - identical members , ( ii ) the size of the ensemble , and ( iii ) the

💾 Sauvegardé : summaries\2605.04039v1_summary.md


---

# OpenSeeker-v2: Pushing the Limits of Search Agents with Informative and High-Difficulty Trajectories
**Auteurs :** Yuwen Du, Rui Ye, Shuo Tang et al.
**Date :** 2026-05-05
**arXiv :** [2605.04036v1](https://arxiv.org/abs/2605.04036v1)

## Sommaire

### Problème abordé
in this paper , we present a new algorithm for the search of the most probable outcome of a search . <n> the algorithm is based on the following three principles : ( 1 ) the algorithm is based on the following three principles : ( 2 ) the algorithm is based on the following three principles : ( 3 ) the algorithm is based on the following three principles : ( 4 ) the algorithm is based on the following three principles : ( 5 ) the algorithm is based on the following three principles : ( 6 ) the algorithm is

### Méthode proposée
we consider the problem of determining whether a biological system is in a state of active or passive evolution . <n> we propose a method to determine whether a biological system is in a state of active or passive evolution . <n> the method is based on the observation that the state of a system is determined by the state of its environment . <n> the state of the system is then determined by the state of the environment . <n> the method is applied to the case of a bacterial population . <n> we show that the search of a large class of biologically active molecules can

### Conclusion / implications
we study the effect of thermal noise on the dynamics of a quantum dot coupled to a thermal bath . <n> we show that the thermal noise can significantly affect the dynamics of the quantum dot . <n> in particular , we find that the temperature of the thermal bath can be increased by increasing the temperature of the quantum dot . <n> we also show that the thermal noise can we present a new method for determining whether or not a given system is in a state of thermal equilibrium . <n> the method is based on the observation that a given system is in a state of

💾 Sauvegardé : summaries\2605.04036v1_summary.md
